# Notebook 14 – Domain-Based Feature Engineering

Domain chosen: Retail


In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


**Code Explanation:** The dataset is loaded, rows without a CustomerID are dropped since anonymous transactions cannot be tied to a customer, InvoiceDate is converted to a real date type, and TotalPrice is calculated for use across the features below.

### Feature 1: TotalPrice

**Feature Name:** TotalPrice

**Source Columns:** Quantity, UnitPrice

**Logic:** Quantity multiplied by UnitPrice.

**Business Meaning:** The actual revenue generated by one line item in an order.

**ML Relevance:** Core numeric feature for any revenue, spend, or value prediction task.

**Leakage Risk:** Low on its own, but high if the target itself is a direct function of TotalPrice, like a High_Value label built from the same value.

**Retain or Remove:** Retain, since it is a fundamental business metric, but never use it alongside a target that was derived directly from it.

In [2]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df[['Quantity', 'UnitPrice', 'TotalPrice']].head()

,Quantity,UnitPrice,TotalPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


### Feature 2: Is_Cancelled

**Feature Name:** Is_Cancelled

**Source Columns:** InvoiceNo

**Logic:** True if InvoiceNo starts with the letter C, which marks a cancelled order in this dataset.

**Business Meaning:** Flags orders that were reversed or returned, which matters for tracking real vs cancelled revenue.

**ML Relevance:** Useful as a target for predicting cancellations, but risky as an input feature for most other tasks.

**Leakage Risk:** High, since cancellation status is often only known after the transaction outcome is already decided.

**Retain or Remove:** Retain only if predicting cancellations directly. Remove from any model predicting revenue, demand, or customer value, since it happens after the fact.

In [3]:
df['Is_Cancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')
df['Is_Cancelled'].value_counts()

Is_Cancelled
False    397924
True       8905
Name: count, dtype: int64

### Feature 3: Day_Of_Week and Is_Weekend

**Feature Name:** Day_Of_Week, Is_Weekend

**Source Columns:** InvoiceDate

**Logic:** Day_Of_Week extracts the weekday name from InvoiceDate. Is_Weekend checks if that day falls on Saturday or Sunday.

**Business Meaning:** Retail demand often shifts by day of week, with weekends behaving differently from weekdays.

**ML Relevance:** Helps models learn weekly demand patterns and seasonality-like effects.

**Leakage Risk:** Low, since the invoice date is always known before the sale happens in a real deployment setting.

**Retain or Remove:** Retain, since it is safe and adds genuine time-based business signal.

In [4]:
df['Day_Of_Week'] = df['InvoiceDate'].dt.day_name()
df['Is_Weekend'] = df['InvoiceDate'].dt.dayofweek >= 5
df[['InvoiceDate', 'Day_Of_Week', 'Is_Weekend']].head()

,InvoiceDate,Day_Of_Week,Is_Weekend
0,2010-12-01 08:26:00,Wednesday,False
1,2010-12-01 08:26:00,Wednesday,False
2,2010-12-01 08:26:00,Wednesday,False
3,2010-12-01 08:26:00,Wednesday,False
4,2010-12-01 08:26:00,Wednesday,False


### Feature 4: Basket_Size

**Feature Name:** Basket_Size

**Source Columns:** InvoiceNo, StockCode

**Logic:** Count of unique products purchased within the same invoice.

**Business Meaning:** Shows how many different items a customer bought in a single shopping trip.

**ML Relevance:** Useful for customer segmentation and predicting order value or purchase behavior.

**Leakage Risk:** Low, since a completed invoice's basket size is known at the time the order is placed.

**Retain or Remove:** Retain, since it reflects genuine shopping behavior available at prediction time.

In [5]:
df['Basket_Size'] = df.groupby('InvoiceNo')['StockCode'].transform('nunique')
df[['InvoiceNo', 'Basket_Size']].head()

,InvoiceNo,Basket_Size
0,536365,7
1,536365,7
2,536365,7
3,536365,7
4,536365,7


### Feature 5: Recency

**Feature Name:** Recency

**Source Columns:** CustomerID, InvoiceDate

**Logic:** Number of days between a snapshot date and each customer's most recent purchase.

**Business Meaning:** Shows how recently a customer has engaged with the business, a core part of RFM analysis.

**ML Relevance:** Strong predictor of churn risk and future purchase likelihood.

**Leakage Risk:** Medium, since it must be calculated using only data available up to the snapshot date, not future purchases.

**Retain or Remove:** Retain, as long as the snapshot date is set correctly relative to when the prediction is actually being made.

In [6]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
customer_features = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days)
).reset_index()
customer_features.head()

,CustomerID,Recency
0,12346.0,326
1,12347.0,2
2,12348.0,75
3,12349.0,19
4,12350.0,310


### Feature 6: Frequency

**Feature Name:** Frequency

**Source Columns:** CustomerID, InvoiceNo

**Logic:** Count of unique invoices per customer.

**Business Meaning:** Shows how often a customer shops, a sign of loyalty and habit.

**ML Relevance:** Useful for segmenting loyal customers versus one-time buyers.

**Leakage Risk:** Low if computed strictly from the training period, high if it includes invoices from after the prediction point.

**Retain or Remove:** Retain, with the same care as Recency about using only past data.

In [7]:
customer_features['Frequency'] = df.groupby('CustomerID')['InvoiceNo'].nunique().values
customer_features.head()

,CustomerID,Recency,Frequency
0,12346.0,326,2
1,12347.0,2,7
2,12348.0,75,4
3,12349.0,19,1
4,12350.0,310,1


### Feature 7: Monetary

**Feature Name:** Monetary

**Source Columns:** CustomerID, TotalPrice

**Logic:** Sum of TotalPrice across all of a customer's transactions.

**Business Meaning:** Total value a customer has brought to the business so far.

**ML Relevance:** Key input for customer lifetime value models and value-based segmentation.

**Leakage Risk:** High if it is also used to build the prediction target, like a High_Value label based on total spend.

**Retain or Remove:** Retain as an input feature, but never pair it with a target that was derived from the same total spend figure.

In [8]:
customer_features['Monetary'] = df.groupby('CustomerID')['TotalPrice'].sum().values
customer_features.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,2,0.00
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40


### Feature 8: Avg_Order_Value

**Feature Name:** Avg_Order_Value

**Source Columns:** Monetary, Frequency

**Logic:** Monetary divided by Frequency.

**Business Meaning:** Average amount a customer spends per order, useful for spotting big spenders who order rarely versus frequent small spenders.

**ML Relevance:** Adds nuance beyond raw totals, helpful for customer value tiering.

**Leakage Risk:** Low, since it is built from already-safe aggregated features, as long as those features themselves are leakage-free.

**Retain or Remove:** Retain, since it is a meaningful ratio that adds insight beyond the two raw numbers alone.

In [9]:
customer_features['Avg_Order_Value'] = customer_features['Monetary'] / customer_features['Frequency']
customer_features.head()

,CustomerID,Recency,Frequency,Monetary,Avg_Order_Value
0,12346.0,326,2,0.00,0.000000
1,12347.0,2,7,4310.00,615.714286
2,12348.0,75,4,1797.24,449.310000
3,12349.0,19,1,1757.55,1757.550000
4,12350.0,310,1,334.40,334.400000


### Feature 9: Repeat_Customer

**Feature Name:** Repeat_Customer

**Source Columns:** Frequency

**Logic:** True if Frequency is greater than 1, meaning the customer has placed more than one order.

**Business Meaning:** Separates one-time buyers from customers who came back, a key loyalty signal.

**ML Relevance:** Simple but effective binary feature for churn and loyalty modeling.

**Leakage Risk:** Medium, since Frequency must only count orders placed before the prediction point, not future ones.

**Retain or Remove:** Retain, as long as Frequency itself is calculated safely using only past data.

In [10]:
customer_features['Repeat_Customer'] = customer_features['Frequency'] > 1
customer_features.head()

,CustomerID,Recency,Frequency,Monetary,Avg_Order_Value,Repeat_Customer
0,12346.0,326,2,0.00,0.000000,True
1,12347.0,2,7,4310.00,615.714286,True
2,12348.0,75,4,1797.24,449.310000,True
3,12349.0,19,1,1757.55,1757.550000,False
4,12350.0,310,1,334.40,334.400000,False


### Feature 10: Product_Popularity

**Feature Name:** Product_Popularity

**Source Columns:** StockCode, InvoiceNo

**Logic:** Count of unique invoices that included a given StockCode.

**Business Meaning:** Shows how in-demand a product is across all customers.

**ML Relevance:** Useful for recommendation systems, inventory planning, and predicting which products will sell well.

**Leakage Risk:** Medium, since including future invoices when calculating popularity for a past prediction date would leak future demand information.

**Retain or Remove:** Retain, but always calculate it using only invoices dated before the prediction point.

In [11]:
product_popularity = df.groupby('StockCode')['InvoiceNo'].nunique().sort_values(ascending=False)
product_popularity.head()

StockCode
85123A    2020
22423     1884
85099B    1643
47566     1399
84879     1385
Name: InvoiceNo, dtype: int64

### Feature 11: Country_Region

**Feature Name:** Country_Region

**Source Columns:** Country

**Logic:** Groups Country into Domestic if it is United Kingdom, otherwise International.

**Business Meaning:** Domestic and international orders often behave differently in terms of shipping cost, order size, and return rates.

**ML Relevance:** A simpler, lower-cardinality alternative to one-hot encoding every individual country.

**Leakage Risk:** Low, since the customer's country is known at order time and does not depend on the outcome being predicted.

**Retain or Remove:** Retain, since it reduces category count while keeping a meaningful business distinction.

In [12]:
df['Country_Region'] = df['Country'].apply(lambda x: 'Domestic' if x == 'United Kingdom' else 'International')
df['Country_Region'].value_counts()

Country_Region
Domestic         361878
International     44951
Name: count, dtype: int64

### Feature 12: Tenure_Days

**Feature Name:** Tenure_Days

**Source Columns:** CustomerID, InvoiceDate

**Logic:** Number of days between a customer's very first purchase and the snapshot date.

**Business Meaning:** Shows how long a customer has been active with the business, distinguishing new customers from long-time ones.

**ML Relevance:** Useful for customer lifetime value modeling and understanding loyalty patterns over time.

**Leakage Risk:** Low, since the first purchase date is always fixed and known once it has happened.

**Retain or Remove:** Retain, since it is a stable, safe, and business-meaningful feature.

In [13]:
first_purchase = df.groupby('CustomerID')['InvoiceDate'].min()
customer_features['Tenure_Days'] = (snapshot_date - customer_features['CustomerID'].map(first_purchase)).dt.days
customer_features.head()

,CustomerID,Recency,Frequency,Monetary,Avg_Order_Value,Repeat_Customer,Tenure_Days
0,12346.0,326,2,0.00,0.000000,True,326
1,12347.0,2,7,4310.00,615.714286,True,367
2,12348.0,75,4,1797.24,449.310000,True,358
3,12349.0,19,1,1757.55,1757.550000,False,19
4,12350.0,310,1,334.40,334.400000,False,310
